# KCV-TTS Finetuning di Kaggle (2x T4, DDP, W&B)
Notebook ini menyiapkan pipeline penuh: venv, install dependency (termasuk `causal-conv1d` dan `mamba-ssm`), clone repo, prepare dataset CSV -> Arrow, lalu finetuning dengan 2 GPU T4 menggunakan `accelerate` + W&B.

## 1) Isi Konfigurasi
Ubah nilai di cell berikut sesuai repo dan path dataset kamu di Kaggle.

In [ ]:
import os
from pathlib import Path

# ===== WAJIB DIISI =====
REPO_URL = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "main"
WANDB_API_KEY = "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"
WANDB_ENTITY = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kcvanguard"

# CSV metadata sumber di Kaggle (header: audio_file|text, path audio absolut)
# Contoh: /kaggle/input/your-dataset/metadata.csv
SRC_METADATA_CSV = "/kaggle/input/datasets/anekazek/indonesian-voice-dataset/datasetku/metadata.csv"

# Nama dataset hasil prep (akan jadi data/{DATASET_NAME}_pinyin)
DATASET_NAME = "datasetku"

# Output training
MODEL_NAME = "F5TTS_MAMBA_kaggle_full"
TARGET_SAMPLE_RATE = 24000

# Training optimization untuk 2x T4
BATCH_SIZE_PER_GPU = 512
MAX_SAMPLES = 2
GRAD_ACCUM = 4
LEARNING_RATE = 1.0e-5
NUM_WARMUP_UPDATES = 200
EPOCHS = 50
NUM_WORKERS = 4

# Path kerja
WORKDIR = Path('/kaggle/temp')
VENV_DIR = WORKDIR / '.venv'
REPO_DIR = WORKDIR / 'kcv-tts'

assert WANDB_API_KEY != "PASTE_WANDB_API_KEY_DI_SINI", "Isi WANDB_API_KEY dulu"
assert REPO_URL != "https://github.com/<username>/<repo>.git", "Isi REPO_URL dulu"
assert Path(SRC_METADATA_CSV).exists(), f"metadata tidak ditemukan: {SRC_METADATA_CSV}"

# Export ke environment agar tersedia di semua cell %%bash
os.environ['REPO_URL'] = REPO_URL
os.environ['REPO_BRANCH'] = REPO_BRANCH
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online'
os.environ['SRC_METADATA_CSV'] = SRC_METADATA_CSV
os.environ['DATASET_NAME'] = DATASET_NAME
os.environ['MODEL_NAME'] = MODEL_NAME
os.environ['TARGET_SAMPLE_RATE'] = str(TARGET_SAMPLE_RATE)
os.environ['BATCH_SIZE_PER_GPU'] = str(BATCH_SIZE_PER_GPU)
os.environ['MAX_SAMPLES'] = str(MAX_SAMPLES)
os.environ['GRAD_ACCUM'] = str(GRAD_ACCUM)
os.environ['LEARNING_RATE'] = str(LEARNING_RATE)
os.environ['NUM_WARMUP_UPDATES'] = str(NUM_WARMUP_UPDATES)
os.environ['EPOCHS'] = str(EPOCHS)
os.environ['NUM_WORKERS'] = str(NUM_WORKERS)

print('Config OK')
print('SRC_METADATA_CSV =', SRC_METADATA_CSV)

In [ ]:
!nvidia-smi

## 2) Build Venv + Clone Repo + Install Dependencies

In [ ]:
%%bash
apt-get update -qq
apt-get install -y python3.11 python3.11-venv python3.11-distutils > /dev/null
python3.11 --version

In [ ]:
%%bash
set -euo pipefail
mkdir -p /kaggle/temp
cd /kaggle/temp

# Isolate from host/site customizations that can break venv bootstrap.
export PYTHONNOUSERSITE=1
unset PYTHONPATH || true
unset PYTHONHOME || true

# Require Python 3.11 to match local ABI/wheels exactly.
if command -v python3.11 >/dev/null 2>&1; then
  PY311=python3.11
else
  echo "python3.11 tidak tersedia di runtime ini. Ganti Kaggle image/runtime yang punya Python 3.11."
  exit 1
fi

# Build venv (fallback to virtualenv if stdlib venv fails).
if ! "$PY311" -m venv .venv; then
  "$PY311" -m pip install --upgrade pip
  "$PY311" -m pip install virtualenv
  "$PY311" -m virtualenv .venv
fi

source .venv/bin/activate
python - <<'PY'
import sys
assert sys.version_info[:2] == (3, 11), f"Butuh Python 3.11, dapat: {sys.version}"
print('python =', sys.version.split()[0])
PY

python -m pip install --upgrade pip setuptools wheel wrapt

# Clone repo
: "${REPO_BRANCH:=main}"
: "${REPO_URL:?REPO_URL is not set. Run Cell 3 (config) first.}"
if [ -d kcv-tts ]; then rm -rf kcv-tts; fi
git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" kcv-tts

cd kcv-tts

# Match stack exactly: torch 2.6/cu124 + project deps
test -f requirements-py311-torch260-cu124.txt
pip install -r requirements-py311-torch260-cu124.txt
pip install -e . --no-deps

In [ ]:
# Requested order: causal first, then mamba
# Requested order: causal first, then mamba
!/kaggle/temp/.venv/bin/pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

In [ ]:
!/kaggle/temp/.venv/bin/pip install https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl

In [ ]:
%%bash
set -e

VENV=/kaggle/temp/.venv

"$VENV/bin/python" -V
"$VENV/bin/python" -c "import sys; print(sys.executable)"
"$VENV/bin/pip" -V

"$VENV/bin/pip" install --force-reinstall numpy==1.26.4 scipy==1.13.1

"$VENV/bin/python" - <<'PY'
from importlib import metadata
import sys

print("PYTHON:", sys.executable)
print("VERSION:", sys.version)

pkgs = [
    'torch','torchaudio','torchvision',
    'numpy','scipy','accelerate','hydra-core',
    'causal-conv1d','mamba-ssm','bitsandbytes'
]
for p in pkgs:
    try:
        print(p, metadata.version(p))
    except metadata.PackageNotFoundError:
        print(p, 'MISSING')
PY

In [ ]:
%%bash
set -euo pipefail
source /kaggle/temp/.venv/bin/activate
python - <<'PY'
import os
import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])
print('W&B login OK')
PY

## 3) Prepare Dataset (CSV -> Arrow)

In [ ]:
import csv
from pathlib import Path

src = Path(SRC_METADATA_CSV)
dst = Path('/kaggle/temp/metadata_abs.csv')

with src.open('r', encoding='utf-8-sig', newline='') as fin, dst.open('w', encoding='utf-8', newline='') as fout:
    r = csv.reader(fin, delimiter='|')
    w = csv.writer(fout, delimiter='|', lineterminator='\n')
    header = next(r, None)
    if not header or header[0].strip() != 'audio_file' or header[1].strip() != 'text':
        raise ValueError('Header CSV harus: audio_file|text')
    w.writerow(['audio_file', 'text'])
    n = 0
    for row in r:
        if len(row) < 2:
            continue
        audio = row[0].strip()
        text = row[1].strip()
        if not audio or not text:
            continue
        p = Path(audio).expanduser()
        if not p.is_absolute():
            # Jika relatif, jadikan relatif terhadap folder CSV sumber
            p = (src.parent / p).resolve()
        if p.exists():
            w.writerow([p.as_posix(), text])
            n += 1

print('metadata_abs.csv rows =', n)
print('saved at', dst)

In [ ]:
%%bash
set -e

TARGET_DIR="/kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin"
TMP_DIR="/kaggle/temp/hf_tmp_emilia"

pip install -q huggingface_hub

python - <<'PY'
from huggingface_hub import snapshot_download
import os, shutil

target = "/kaggle/temp/kcv-tts/data/Emilia_ZH_EN_pinyin"
tmpdir = "/kaggle/temp/hf_tmp_emilia"

snapshot_download(
    repo_id="mrfakename/E2-F5-TTS",
    repo_type="space",
    revision="27cee60c0890d22dab124730a73d5453fc8359a5",
    allow_patterns=["data/Emilia_ZH_EN_pinyin/*"],
    local_dir=tmpdir,
    local_dir_use_symlinks=False,
)

src = os.path.join(tmpdir, "data", "Emilia_ZH_EN_pinyin")
os.makedirs(target, exist_ok=True)

for name in os.listdir(src):
    s = os.path.join(src, name)
    d = os.path.join(target, name)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("done:", target)
PY

ls -lah "$TARGET_DIR"
test -f "$TARGET_DIR/vocab.txt"

In [ ]:
%%bash
set -euxo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

VENV_PY="/kaggle/temp/.venv/bin/python"
DATASET_NAME="${DATASET_NAME:-datasetku}"
META_CSV="/kaggle/temp/metadata_abs.csv"
OUT_DIR="/kaggle/temp/kcv-tts/data/${DATASET_NAME}_pinyin"

test -x "$VENV_PY"
test -f "$META_CSV"

echo "DATASET_NAME=$DATASET_NAME"
echo "META_CSV=$META_CSV"
echo "OUT_DIR=$OUT_DIR"
head -n 3 "$META_CSV"

"$VENV_PY" src/f5_tts/train/datasets/prepare_csv_wavs.py "$META_CSV" "$OUT_DIR" --workers 8
ls -lah "$OUT_DIR"

## 4) Download Pretrained Checkpoint ke Save Dir (untuk Finetune)

In [ ]:
%%bash
set -euxo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

DATASET_NAME="${DATASET_NAME:-datasetku}"
MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
CKPT_DIR="/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
mkdir -p "$CKPT_DIR"

python - <<'PY'
from pathlib import Path
import os
import shutil

from huggingface_hub import hf_hub_download

# Force Indonesian warm-start assets from this repo
repo_id = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
ckpt_filename = "f5_tts_indo_v2.pt"
vocab_filename = "vocab.txt"

model_name = os.environ.get("MODEL_NAME", "F5TTS_MAMBA_kaggle_full")
dataset_name = os.environ.get("DATASET_NAME", "datasetku")
ckpt_dir = Path(f"/kaggle/working/ckpts/{model_name}_vocos_pinyin_{dataset_name}")
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Trainer auto-detects pretrained_* when no model_*.pt exists
dst_pt = ckpt_dir / "pretrained_indo_v2.pt"
src_pt = hf_hub_download(repo_id=repo_id, filename=ckpt_filename)
shutil.copy2(src_pt, dst_pt)

dst_vocab = ckpt_dir / "vocab.txt"
src_vocab = hf_hub_download(repo_id=repo_id, filename=vocab_filename)
shutil.copy2(src_vocab, dst_vocab)

print("repo:", repo_id)
print("checkpoint ready:", dst_pt)
print("vocab ready:", dst_vocab)
PY

ls -lah "$CKPT_DIR"

## 5) Finetune Full di 2x T4 (DDP) + W&B

In [ ]:
# %%bash
# set -euo pipefail
# source /kaggle/temp/.venv/bin/activate

# python - <<'PY'
# import torch
# print("torch:", torch.__version__, "cuda:", torch.version.cuda)
# print("cxx11abi:", torch._C._GLIBCXX_USE_CXX11_ABI)
# PY

# pip uninstall -y causal-conv1d mamba-ssm || true
# pip cache purge || true

# # 1) Coba wheel yang match torch2.6 + cu12 + cp311 + cxx11abiTRUE
# pip install -U --no-deps \
#   https://github.com/state-spaces/mamba/releases/download/v2.3.1/causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl \
#   https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.6cxx11abiTRUE-cp311-cp311-linux_x86_64.whl \
# || {
#   echo "[warn] direct wheel install gagal, fallback ke PyPI build"
#   pip install -U --no-build-isolation causal-conv1d==1.5.0.post8 mamba-ssm==2.3.1
# }

# python - <<'PY'
# import torch, causal_conv1d, mamba_ssm
# print("OK torch", torch.__version__)
# print("OK causal_conv1d", getattr(causal_conv1d, "__version__", "n/a"))
# print("OK mamba_ssm", getattr(mamba_ssm, "__version__", "n/a"))
# PY

In [ ]:
# %%bash
# set -euo pipefail
# source /kaggle/temp/.venv/bin/activate

# echo "[A] cek torch ABI"
# python - <<'PY'
# import torch
# print("torch:", torch.__version__, "cuda:", torch.version.cuda, "cxx11abi:", torch._C._GLIBCXX_USE_CXX11_ABI)
# PY

# echo "[B] uninstall + bersihkan sisa file"
# pip uninstall -y causal-conv1d mamba-ssm || true
# python - <<'PY'
# import site, glob, os, shutil
# for d in site.getsitepackages():
#     for p in glob.glob(os.path.join(d, "causal_conv1d*")) + glob.glob(os.path.join(d, "mamba_ssm*")):
#         if os.path.isdir(p):
#             shutil.rmtree(p, ignore_errors=True)
#         else:
#             try: os.remove(p)
#             except: pass
#         print("removed:", p)
# PY
# pip cache purge || true

# echo "[C] toolchain"
# apt-get update -y
# apt-get install -y build-essential python3.11-dev

# echo "[D] build ulang paksa ABI FALSE"
# export CXXFLAGS="-D_GLIBCXX_USE_CXX11_ABI=0"
# pip install --no-cache-dir --no-build-isolation --no-binary=:all: causal-conv1d==1.5.0.post8
# pip install --no-cache-dir --no-build-isolation --no-binary=:all: mamba-ssm==2.3.1

# echo "[E] verifikasi import"
# python - <<'PY'
# import torch, causal_conv1d, mamba_ssm
# print("OK torch", torch.__version__, "abi", torch._C._GLIBCXX_USE_CXX11_ABI)
# print("OK causal_conv1d", getattr(causal_conv1d, "__version__", "n/a"))
# print("OK mamba_ssm", getattr(mamba_ssm, "__version__", "n/a"))
# PY

In [ ]:
!apt-get update -y
!apt-get install -y build-essential python3.11-dev

In [ ]:
# %%bash
# set -Eeuo pipefail

# on_err() {
#   ec=$?
#   echo ""
#   echo "[ERROR] line=$1 exit=$ec"
#   if [ -f /kaggle/working/train_kaggle.log ]; then
#     echo "----- tail /kaggle/working/train_kaggle.log -----"
#     tail -n 200 /kaggle/working/train_kaggle.log || true
#   fi
#   exit "$ec"
# }
# trap 'on_err $LINENO' ERR

# source /kaggle/temp/.venv/bin/activate
# cd /kaggle/temp/kcv-tts

# export HYDRA_FULL_ERROR=1
# export TORCH_DISTRIBUTED_DEBUG=DETAIL
# export PYTHONUNBUFFERED=1

# DATASET_NAME="${DATASET_NAME:-datasetku}"
# MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
# BATCH_SIZE_PER_GPU="${BATCH_SIZE_PER_GPU:-256}"
# MAX_SAMPLES="${MAX_SAMPLES:-1}"
# NUM_WORKERS="${NUM_WORKERS:-2}"
# EPOCHS="${EPOCHS:-1}"
# LEARNING_RATE="${LEARNING_RATE:-1e-5}"
# NUM_WARMUP_UPDATES="${NUM_WARMUP_UPDATES:-20}"
# GRAD_ACCUM="${GRAD_ACCUM:-4}"
# TARGET_SAMPLE_RATE="${TARGET_SAMPLE_RATE:-24000}"
# WANDB_PROJECT="${WANDB_PROJECT:-kcvanguard}"

# LOG_FILE=/kaggle/working/train_kaggle.log
# : > "$LOG_FILE"

# echo "[1/6] runtime check"
# which python
# python -V
# which accelerate
# accelerate launch --help >/dev/null

# echo "[2/6] dependency check (core only)"
# python - <<'PY'
# import importlib, torch
# print("torch", torch.__version__, "cuda", torch.version.cuda, "cxx11abi", torch._C._GLIBCXX_USE_CXX11_ABI)
# for m in ["accelerate", "hydra"]:
#     mod = importlib.import_module(m)
#     print("OK", m, getattr(mod, "__version__", "n/a"))
# print("deps_core_ok")
# PY

# echo "[3/6] mamba optional check"
# if python - <<'PY'
# import importlib, sys
# try:
#     importlib.import_module("causal_conv1d")
#     importlib.import_module("mamba_ssm")
#     print("mamba_stack_ok")
#     sys.exit(0)
# except Exception as e:
#     print("mamba_stack_off:", e)
#     sys.exit(1)
# PY
# then
#   USE_MAMBA=true
# else
#   USE_MAMBA=false
# fi
# echo "USE_MAMBA=$USE_MAMBA"

# echo "[4/6] dataset + pretrained"
# DATA_DIR="data/${DATASET_NAME}_pinyin"
# test -d "$DATA_DIR"

# CKPT_DIR="ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
# PRETRAIN_FILE="${CKPT_DIR}/pretrained_model_1250000.safetensors"

# if [ ! -f "$PRETRAIN_FILE" ]; then
#   ALT_PRETRAIN="$(ls -1 ckpts/*_vocos_pinyin_${DATASET_NAME}/pretrained_*.safetensors 2>/dev/null | head -n 1 || true)"
#   if [ -n "$ALT_PRETRAIN" ]; then
#     mkdir -p "$CKPT_DIR"
#     cp -f "$ALT_PRETRAIN" "$PRETRAIN_FILE"
#     echo "fallback pretrained copied: $ALT_PRETRAIN -> $PRETRAIN_FILE"
#   fi
# fi
# test -f "$PRETRAIN_FILE"

# echo "[5/6] logger + gpu"
# if [ -n "${WANDB_API_KEY:-}" ]; then
#   export WANDB_API_KEY
#   export WANDB_ENTITY="${WANDB_ENTITY:-}"
#   export WANDB_PROJECT
#   export WANDB_MODE=online
#   LOGGER_ARGS=(++ckpts.logger=wandb "++ckpts.wandb_project=${WANDB_PROJECT}" "++ckpts.wandb_run_name=${MODEL_NAME}-kaggle")
# else
#   export WANDB_MODE=offline
#   LOGGER_ARGS=(++ckpts.logger=tensorboard)
# fi

# GPU_COUNT="$(python - <<'PY'
# import torch
# print(torch.cuda.device_count())
# PY
# )"
# echo "GPU_COUNT=$GPU_COUNT"

# if [ "$GPU_COUNT" -ge 2 ]; then
#   export CUDA_VISIBLE_DEVICES=0,1
#   DIST_ARGS=(--multi_gpu --num_processes 2)
# elif [ "$GPU_COUNT" -eq 1 ]; then
#   export CUDA_VISIBLE_DEVICES=0
#   DIST_ARGS=(--num_processes 1)
# else
#   echo "No GPU detected"
#   exit 1
# fi

# if [ "$USE_MAMBA" = true ]; then
#   MODEL_ARGS=(++model.arch.use_mamba=true "++model.arch.mamba_layers=[2,3,6,7]")
# else
#   MODEL_ARGS=(++model.arch.use_mamba=false)
# fi

# echo "[6/6] training"
# set +e
# accelerate launch "${DIST_ARGS[@]}" --mixed_precision=fp16 \
#   src/f5_tts/train/train.py \
#   --config-name F5TTS_SANITY_HYBRID_1K.yaml \
#   "++datasets.name=${DATASET_NAME}" \
#   "++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU}" \
#   ++datasets.batch_size_type=frame \
#   "++datasets.max_samples=${MAX_SAMPLES}" \
#   "++datasets.num_workers=${NUM_WORKERS}" \
#   "++optim.epochs=${EPOCHS}" \
#   "++optim.learning_rate=${LEARNING_RATE}" \
#   "++optim.num_warmup_updates=${NUM_WARMUP_UPDATES}" \
#   "++optim.grad_accumulation_steps=${GRAD_ACCUM}" \
#   ++optim.max_grad_norm=1.0 \
#   ++optim.bnb_optimizer=False \
#   ++ckpts.log_samples=False \
#   ++ckpts.save_per_updates=200 \
#   ++ckpts.last_per_updates=100 \
#   ++ckpts.keep_last_n_checkpoints=3 \
#   "++model.name=${MODEL_NAME}" \
#   ++model.tokenizer=pinyin \
#   "++model.mel_spec.target_sample_rate=${TARGET_SAMPLE_RATE}" \
#   ++model.arch.dim=512 \
#   ++model.arch.depth=10 \
#   ++model.arch.heads=8 \
#   ++model.arch.text_dim=512 \
#   ++model.arch.checkpoint_activations=True \
#   "${MODEL_ARGS[@]}" \
#   "${LOGGER_ARGS[@]}" \
#   2>&1 | tee "$LOG_FILE"
# RC=${PIPESTATUS[0]}
# set -e

# if [ "$RC" -ne 0 ]; then
#   echo "Training gagal RC=$RC"
#   tail -n 200 "$LOG_FILE" || true
#   exit "$RC"
# fi

# echo "Training selesai RC=0"
# find "$CKPT_DIR" -maxdepth 3 -type f -name "model*.pt" -print | sort || true

In [ ]:
%%bash
set -Eeuo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

export HYDRA_FULL_ERROR=1
export TORCH_DISTRIBUTED_DEBUG=DETAIL
export PYTHONUNBUFFERED=1
export CUDA_VISIBLE_DEVICES=0,1

# Training defaults (override from Cell 3 env if needed)
DATASET_NAME="${DATASET_NAME:-datasetku}"
MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
WANDB_PROJECT="${WANDB_PROJECT:-kcvanguard}"

BATCH_SIZE_PER_GPU="${BATCH_SIZE_PER_GPU:-512}"
MAX_SAMPLES="${MAX_SAMPLES:-2}"
NUM_WORKERS="${NUM_WORKERS:-4}"
EPOCHS="${EPOCHS:-10}"
LEARNING_RATE="${LEARNING_RATE:-1e-5}"
NUM_WARMUP_UPDATES="${NUM_WARMUP_UPDATES:-200}"
GRAD_ACCUM="${GRAD_ACCUM:-4}"

# Save checkpoints to persistent storage
CKPT_ROOT="/kaggle/working/ckpts"
CKPT_DIR="${CKPT_ROOT}/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
mkdir -p "$CKPT_DIR"

# Enforce warm-start from HF Indonesian checkpoint
PRETRAIN_PT="${CKPT_DIR}/pretrained_indo_v2.pt"
if [ ! -f "$PRETRAIN_PT" ]; then
  echo "Missing pretrained checkpoint: $PRETRAIN_PT"
  echo "Run Cell 17 (download pretrained) first."
  exit 1
fi

# Remove old model checkpoints so trainer loads pretrained_indo_v2.pt instead of model_last.pt
FORCE_FRESH_FROM_PRETRAIN="${FORCE_FRESH_FROM_PRETRAIN:-1}"
if [ "$FORCE_FRESH_FROM_PRETRAIN" = "1" ]; then
  find "$CKPT_DIR" -maxdepth 1 -type f -name 'model_*.pt' -delete || true
  rm -f "$CKPT_DIR/model_last.pt" || true
  echo "Removed old model_*.pt/model_last.pt (fresh start from pretrained_indo_v2.pt)"
fi

# Keep tokenizer as pinyin for train.py dataset loader.
# train.py couples dataset path to tokenizer: data/{datasets.name}_{model.tokenizer}.
DATA_DIR="data/${DATASET_NAME}_pinyin"
DATA_RAW_ARROW="${DATA_DIR}/raw.arrow"
DATA_RAW_DIR="${DATA_DIR}/raw"
DATA_VOCAB="${DATA_DIR}/vocab.txt"
if [ ! -f "$DATA_RAW_ARROW" ] && [ ! -d "$DATA_RAW_DIR" ]; then
  echo "Dataset not found: need ${DATA_RAW_ARROW} or ${DATA_RAW_DIR}"
  exit 1
fi

# Sync Indo vocab from CKPT_DIR into dataset vocab path used by pinyin tokenizer.
if [ -f "${CKPT_DIR}/vocab.txt" ]; then
  cp -f "${CKPT_DIR}/vocab.txt" "$DATA_VOCAB"
  echo "Tokenizer vocab synced: ${CKPT_DIR}/vocab.txt -> ${DATA_VOCAB}"
fi
TOKENIZER_ARGS=(++model.tokenizer=pinyin)

echo "Disk status before train:"
df -h /kaggle/temp /kaggle/working || true

python - <<'PY'
import torch, accelerate, hydra
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpu_count", torch.cuda.device_count())
print("accelerate", accelerate.__version__)
print("hydra", hydra.__version__)
PY

# Mamba optional fallback
if python - <<'PY'
import importlib, sys
try:
    importlib.import_module("causal_conv1d")
    importlib.import_module("mamba_ssm")
    sys.exit(0)
except Exception:
    sys.exit(1)
PY
then
  MODEL_ARGS=(++model.arch.use_mamba=true "++model.arch.mamba_layers=[2,3,6,7]")
  echo "USE_MAMBA=true"
else
  MODEL_ARGS=(++model.arch.use_mamba=false)
  echo "USE_MAMBA=false"
fi

if [ -n "${WANDB_API_KEY:-}" ]; then
  export WANDB_API_KEY
  export WANDB_ENTITY="${WANDB_ENTITY:-}"
  export WANDB_PROJECT
  export WANDB_MODE=online
  echo "Using logger: wandb"
  LOGGER_ARGS=(++ckpts.logger=wandb "++ckpts.wandb_project=${WANDB_PROJECT}" "++ckpts.wandb_run_name=${MODEL_NAME}")
else
  export WANDB_MODE=offline
  echo "Using logger: tensorboard"
  LOGGER_ARGS=(++ckpts.logger=tensorboard)
fi

# Disable periodic checkpointing; keep only final model_last.pt at end of training.
NO_PERIODIC_SAVE_UPDATES=999999999

accelerate launch --multi_gpu --num_processes 2 --mixed_precision=fp16 \
  src/f5_tts/train/train.py \
  --config-name F5TTS_SANITY_HYBRID_1K.yaml \
  "++datasets.name=${DATASET_NAME}" \
  "++datasets.batch_size_per_gpu=${BATCH_SIZE_PER_GPU}" \
  ++datasets.batch_size_type=frame \
  "++datasets.max_samples=${MAX_SAMPLES}" \
  "++datasets.num_workers=${NUM_WORKERS}" \
  "++optim.epochs=${EPOCHS}" \
  "++optim.learning_rate=${LEARNING_RATE}" \
  "++optim.num_warmup_updates=${NUM_WARMUP_UPDATES}" \
  "++optim.grad_accumulation_steps=${GRAD_ACCUM}" \
  ++optim.max_grad_norm=1.0 \
  ++optim.bnb_optimizer=False \
  ++ckpts.log_samples=False \
  "++ckpts.save_per_updates=${NO_PERIODIC_SAVE_UPDATES}" \
  "++ckpts.last_per_updates=${NO_PERIODIC_SAVE_UPDATES}" \
  ++ckpts.keep_last_n_checkpoints=0 \
  "++ckpts.save_dir=${CKPT_DIR}" \
  "++model.name=${MODEL_NAME}" \
  "${TOKENIZER_ARGS[@]}" \
  "${MODEL_ARGS[@]}" \
  "${LOGGER_ARGS[@]}"

echo "Done: ${CKPT_DIR}"
find "${CKPT_DIR}" -maxdepth 2 -type f -name 'model*.pt' -print | sort || true

## 6) Lokasi Checkpoint dan Quick Inference (Opsional)

In [ ]:
from pathlib import Path
import glob

candidates = [
    Path(f'/kaggle/working/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}'),
    Path(f'/kaggle/temp/kcv-tts/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}'),
    Path(f'/kaggle/temp/kcv-tts/kaggle/working/ckpts/{MODEL_NAME}_vocos_pinyin_{DATASET_NAME}'),
]

print('Candidate checkpoint dirs:')
for d in candidates:
    print('-', d, 'exists=', d.exists())
    pts = sorted(d.glob('model*.pt'))
    for p in pts[-5:]:
        print('  ->', p)

# Broad fallback scan
fallback_patterns = [
    '/kaggle/temp/kcv-tts/**/model_last.pt',
    '/kaggle/temp/kcv-tts/**/model_*.pt',
]

found = []
for pat in fallback_patterns:
    found.extend(glob.glob(pat, recursive=True))

found = sorted(set(found))
print('Fallback model checkpoints found:', len(found))
for p in found[-20:]:
    print('fallback ->', p)

In [ ]:
%%bash
set -euo pipefail
source /kaggle/temp/.venv/bin/activate

mkdir -p /kaggle/working/ref_audio

python - <<'PY'
from huggingface_hub import hf_hub_download
import shutil

repo_id = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
files = [
    "ref_prabowo.mp3",
    "ref_reporter.mp3",
    "ref_windah.mp3",
]

for name in files:
    src = hf_hub_download(repo_id=repo_id, filename=name)
    dst = f"/kaggle/working/ref_audio/{name}"
    shutil.copy2(src, dst)
    print("downloaded:", dst)
PY

ls -lah /kaggle/working/ref_audio

In [ ]:
%%bash
set -euo pipefail
source /kaggle/temp/.venv/bin/activate
cd /kaggle/temp/kcv-tts

# Avoid notebook inline backend leaking into CLI process.
unset MPLBACKEND || true
export MPLBACKEND=Agg

# Contoh quick inference setelah model_last.pt tersedia.
PRIMARY_CKPT_DIR="/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_1="/kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_2="/kaggle/temp/kcv-tts/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"

# Fixed reference inputs requested
REF_AUDIO="/kaggle/working/ref_audio/ref_prabowo.mp3"
REF_TEXT="Selamat pagi, salam sejahtera bagi kita sekalian"

if [ ! -f "$REF_AUDIO" ]; then
  echo "Reference audio tidak ditemukan: $REF_AUDIO"
  echo "Jalankan dulu cell download reference audio."
  exit 1
fi

# Find model_last.pt from explicit and fallback locations.
MODEL_PT=""
for p in \
  "${PRIMARY_CKPT_DIR}/model_last.pt" \
  "${REL_CKPT_DIR_1}/model_last.pt" \
  "${REL_CKPT_DIR_2}/model_last.pt" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_*_${DATASET_NAME}/model_last.pt 2>/dev/null | head -n 1 || true)" \
  "$(find /kaggle/temp/kcv-tts -type f -name model_last.pt 2>/dev/null | tail -n 1 || true)"
do
  if [ -n "$p" ] && [ -f "$p" ]; then
    MODEL_PT="$p"
    break
  fi
done

if [ -z "$MODEL_PT" ]; then
  echo "model_last.pt tidak ditemukan."
  echo "Cek cepat:"
  echo "  find /kaggle/temp/kcv-tts -type f -name 'model*.pt' | tail -n 50"
  exit 1
fi

# Derive checkpoint dir from model path to search nearby hydra config first.
MODEL_DIR="$(dirname "$MODEL_PT")"

# Find Hydra config from several possible training output locations.
MODEL_CFG=""
for p in \
  "$(ls -1t "${MODEL_DIR}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t "${PRIMARY_CKPT_DIR}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t "${REL_CKPT_DIR_1}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t "${REL_CKPT_DIR_2}"/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_*_${DATASET_NAME}/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/*_${DATASET_NAME}/*/.hydra/config.yaml 2>/dev/null | head -n 1 || true)"
do
  if [ -n "$p" ] && [ -f "$p" ]; then
    MODEL_CFG="$p"
    break
  fi
done

# Fallback: synthesize inference config from base training config when hydra output is missing.
if [ -z "$MODEL_CFG" ]; then
  echo "Hydra config tidak ditemukan, membuat fallback config dari base yaml..."
  MODEL_CFG="/kaggle/temp/infer_model_cfg.yaml"
  MODEL_PT="$MODEL_PT" MODEL_CFG="$MODEL_CFG" python - <<'PY'
import os
import torch
from omegaconf import OmegaConf

model_pt = os.environ["MODEL_PT"]
model_cfg_out = os.environ["MODEL_CFG"]
base_cfg = "src/f5_tts/configs/F5TTS_SANITY_HYBRID_1K.yaml"

cfg = OmegaConf.load(base_cfg)
ckpt = torch.load(model_pt, map_location="cpu", weights_only=True)
state = ckpt.get("ema_model_state_dict", ckpt.get("model_state_dict", {}))
keys = list(state.keys())
use_mamba = any("mamba" in k for k in keys)

cfg.model.arch.use_mamba = bool(use_mamba)
if not use_mamba:
    cfg.model.arch.mamba_layers = []

# Align identifiers used in this notebook run.
cfg.model.name = os.environ.get("MODEL_NAME", cfg.model.name)
cfg.datasets.name = os.environ.get("DATASET_NAME", cfg.datasets.name)
cfg.model.tokenizer = "pinyin"

OmegaConf.save(cfg, model_cfg_out)
print("fallback cfg saved:", model_cfg_out)
print("detected use_mamba:", use_mamba)
PY
fi

if [ ! -f "$MODEL_CFG" ]; then
  echo "Config yaml tetap tidak ditemukan/terbuat."
  exit 1
fi

echo "Using model: $MODEL_PT"
echo "Using config: $MODEL_CFG"
echo "Using ref audio: $REF_AUDIO"

python src/f5_tts/infer/infer_cli.py \
  -m F5TTS_Small \
  -mc "$MODEL_CFG" \
  -p "$MODEL_PT" \
  -v data/${DATASET_NAME}_pinyin/vocab.txt \
  -r "$REF_AUDIO" \
  -s "$REF_TEXT" \
  -t "Halo semua kenalin Aku Sedang Makan" \
  -o /kaggle/working \
  -w infer_kaggle.wav \
  --device cuda --nfe_step 24 --cfg_strength 2.0

In [ ]:
%%bash
set -euo pipefail

MODEL_NAME="${MODEL_NAME:-F5TTS_MAMBA_kaggle_full}"
DATASET_NAME="${DATASET_NAME:-datasetku}"

PRIMARY_CKPT_DIR="/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_1="/kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"
REL_CKPT_DIR_2="/kaggle/temp/kcv-tts/kaggle/working/ckpts/${MODEL_NAME}_vocos_pinyin_${DATASET_NAME}"

# Cari model_last.pt dari lokasi utama + fallback.
SRC_MODEL=""
for p in \
  "${PRIMARY_CKPT_DIR}/model_last.pt" \
  "${REL_CKPT_DIR_1}/model_last.pt" \
  "${REL_CKPT_DIR_2}/model_last.pt" \
  "$(ls -1t /kaggle/temp/kcv-tts/ckpts/${MODEL_NAME}_*_${DATASET_NAME}/model_last.pt 2>/dev/null | head -n 1 || true)" \
  "$(find /kaggle/temp/kcv-tts -type f -name model_last.pt 2>/dev/null | tail -n 1 || true)"
do
  if [ -n "$p" ] && [ -f "$p" ]; then
    SRC_MODEL="$p"
    break
  fi
done

if [ -z "$SRC_MODEL" ]; then
  echo "model_last.pt tidak ditemukan di semua lokasi kandidat."
  echo "Debug cepat:"
  echo "  find /kaggle/temp/kcv-tts -type f -name 'model*.pt' | tail -n 50"
  exit 1
fi

DST_DIR="/kaggle/working"
DST_MODEL="${DST_DIR}/model_last_finetuned.pt"
DST_VOCAB="${DST_DIR}/vocab_${DATASET_NAME}.txt"

cp -vf "$SRC_MODEL" "$DST_MODEL"
echo "Model copied to: $DST_MODEL"

SRC_DIR="$(dirname "$SRC_MODEL")"
if [ -f "${SRC_DIR}/vocab.txt" ]; then
  cp -vf "${SRC_DIR}/vocab.txt" "$DST_VOCAB"
  echo "Vocab copied to: $DST_VOCAB"
elif [ -f "${PRIMARY_CKPT_DIR}/vocab.txt" ]; then
  cp -vf "${PRIMARY_CKPT_DIR}/vocab.txt" "$DST_VOCAB"
  echo "Vocab copied from primary ckpt dir: $DST_VOCAB"
fi

echo ""
echo "Files ready in /kaggle/working:"
ls -lh "$DST_MODEL" || true
ls -lh "$DST_VOCAB" || true
ls -1 /kaggle/working/*.pt /kaggle/working/*.wav 2>/dev/null | head -n 50